# Prosodic-Based Vishing Detector

## Imports and Configurations

In [ ]:
from pathlib import Path
import sys
import os

import numpy as np
import pandas as pd
import joblib
import torch
from torch import nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from joblib import Parallel, delayed
from tqdm import tqdm
from tqdm_joblib import tqdm_joblib

from sklearn.preprocessing import RobustScaler

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from modules.dataset_processing import extract_archives, prepare_all_split_dataframes, subsample_by_class
from modules.audio_processing import extract_prosodic_features, PROSODIC_FEATURE_COLUMNS
from modules.models import ProsodyMLP, ProsodyMLPConfig
from modules.attacks import FGSMAttack, FGSMAttackConfig

In [ ]:
DATA_ROOT = PROJECT_ROOT / 'ASVspoof5'
MODEL_DIR = PROJECT_ROOT / 'models'
CHECKPOINT_DIR = MODEL_DIR / 'prosodic'
CACHE_DIR = DATA_ROOT / 'feature_cache'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

TEST_MODE = False
BONAFIDE_SUBSAMPLE = 15000
SPOOF_SUBSAMPLE = 15000
BATCH_SIZE = 256
NUM_EPOCHS = 60
EARLY_STOPPING_PATIENCE = 30
MIN_DELTA = 1e-4
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
RANDOM_SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_JOBS = max(1, os.cpu_count() - 2)

torch.manual_seed(RANDOM_SEED)
print({'device': str(DEVICE), 'data_root': str(DATA_ROOT), 'checkpoint_dir': str(CHECKPOINT_DIR)})

{'device': 'cpu', 'data_root': '/Users/dundale/Downloads/github/Vishing-Detection-Adversarial-Model/ASVspoof5', 'checkpoint_dir': '/Users/dundale/Downloads/github/Vishing-Detection-Adversarial-Model/models/prosodic'}


## Data Preprocessing

In [ ]:
extract_archives(DATA_ROOT)
split_frames = prepare_all_split_dataframes(DATA_ROOT, require_existing_files=True)

if TEST_MODE:
    split_frames['train'] = subsample_by_class(split_frames['train'], BONAFIDE_SUBSAMPLE, SPOOF_SUBSAMPLE, random_seed=RANDOM_SEED)
    split_frames['dev'] = subsample_by_class(split_frames['dev'], BONAFIDE_SUBSAMPLE, SPOOF_SUBSAMPLE, random_seed=RANDOM_SEED)
    split_frames['eval'] = subsample_by_class(split_frames['eval'], BONAFIDE_SUBSAMPLE, SPOOF_SUBSAMPLE, random_seed=RANDOM_SEED)

for split_name, frame in split_frames.items():
    print(split_name, frame.shape, frame['LABEL'].value_counts().to_dict())

/Users/dundale/Downloads/github/Vishing-Detection-Adversarial-Model/ASVspoof5/flac_T
/Users/dundale/Downloads/github/Vishing-Detection-Adversarial-Model/ASVspoof5/flac_T already exists.
/Users/dundale/Downloads/github/Vishing-Detection-Adversarial-Model/ASVspoof5/flac_D
/Users/dundale/Downloads/github/Vishing-Detection-Adversarial-Model/ASVspoof5/flac_D already exists.
/Users/dundale/Downloads/github/Vishing-Detection-Adversarial-Model/ASVspoof5/flac_E_eval
/Users/dundale/Downloads/github/Vishing-Detection-Adversarial-Model/ASVspoof5/flac_E_eval already exists.
train (18801, 13) {1: 15000, 0: 3801}
dev (24041, 13) {1: 15000, 0: 9041}
eval (23410, 13) {1: 15000, 0: 8410}


## Prosodic Feature Extraction

In [ ]:
import shutil

CLEAR_FEATURE_CACHE = True  # Set to True to force rebuild, False to reuse

if CLEAR_FEATURE_CACHE and CACHE_DIR.exists():
    shutil.rmtree(CACHE_DIR)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Cleared feature cache at {CACHE_DIR}")
elif not CACHE_DIR.exists():
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Created feature cache at {CACHE_DIR}")
else:
    print(f"Using existing cache at {CACHE_DIR}")

print(f"Feature count: {len(PROSODIC_FEATURE_COLUMNS)}")

Cleared feature cache at /Users/dundale/Downloads/github/Vishing-Detection-Adversarial-Model/ASVspoof5/feature_cache
Feature count: 49


In [33]:
# Feature caching: compute once, load from disk on reruns

def get_features(file_path):
    safe_name = file_path.replace("/", "_").replace(":", "_")
    cache_path = str(CACHE_DIR / (safe_name + ".pkl"))

    if os.path.exists(cache_path):
        return joblib.load(cache_path)

    feats = extract_prosodic_features(file_path)
    joblib.dump(feats, cache_path)
    return feats


def build_feature_df(df, split_name):
    records = []
    for _, row in tqdm(df.iterrows(), desc=f"Extracting features for {split_name}", total=len(df)):
        feats = get_features(row["FULL_FILE_PATH"])
        feats["ATTACK_LABEL"] = row["ATTACK_LABEL"]
        feats["LABEL"] = row["LABEL"]
        records.append(feats)
    return pd.DataFrame(records)


train_feats = build_feature_df(split_frames['train'], 'train')
dev_feats = build_feature_df(split_frames['dev'], 'dev')
eval_feats = build_feature_df(split_frames['eval'], 'eval')

Extracting features for dev:  35%|███▍      | 8310/24041 [05:01<08:17, 31.63it/s]/Users/dundale/Downloads/github/Vishing-Detection-Adversarial-Model/.venv/lib/python3.13/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1360
  warnings.warn(
Extracting features for eval: 100%|██████████| 23410/23410 [13:48<00:00, 28.24it/s]


In [ ]:
# Build scaled feature matrices

for df in [train_feats, dev_feats, eval_feats]:
    df[PROSODIC_FEATURE_COLUMNS] = df[PROSODIC_FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)


train_medians = train_feats[PROSODIC_FEATURE_COLUMNS].median()


for df_name, df in [("train", train_feats), ("dev", dev_feats), ("eval", eval_feats)]:
    df[PROSODIC_FEATURE_COLUMNS] = df[PROSODIC_FEATURE_COLUMNS].fillna(train_medians)


X_train = train_feats[PROSODIC_FEATURE_COLUMNS].values.astype(np.float32)
y_train = train_feats["LABEL"].values.astype(np.int64)

X_dev = dev_feats[PROSODIC_FEATURE_COLUMNS].values.astype(np.float32)
y_dev = dev_feats["LABEL"].values.astype(np.int64)

X_eval = eval_feats[PROSODIC_FEATURE_COLUMNS].values.astype(np.float32)
y_eval = eval_feats["LABEL"].values.astype(np.int64)

scaler = RobustScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_dev = scaler.transform(X_dev).astype(np.float32)
X_eval = scaler.transform(X_eval).astype(np.float32)

# DataLoaders
train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
dev_dataset = TensorDataset(torch.tensor(X_dev), torch.tensor(y_dev))
eval_dataset = TensorDataset(torch.tensor(X_eval), torch.tensor(y_eval))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE, shuffle=False)
eval_loader = DataLoader(eval_dataset, batch_size=BATCH_SIZE, shuffle=False)

## Training

In [ ]:
model_config = ProsodyMLPConfig(
    input_dim=len(PROSODIC_FEATURE_COLUMNS),  # will be larger now
    num_classes=2,
    hidden_dims=(512, 384, 256, 128, 64),   # added one layer
    dropout=0.25,                            # slightly lower
    noise_std=0.05,                          # reduced a bit
    use_attention=True,
)
model = ProsodyMLP(config=model_config).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6)
class_weights = torch.tensor([5.0, 1.0]).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
model

ProsodyMLP(
  (hidden): Sequential(
    (0): Linear(in_features=49, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): GELU(approximate='none')
    (7): Dropout(p=0.5, inplace=False)
  )
  (head): Linear(in_features=64, out_features=2, bias=True)
  (residual_proj): Linear(in_features=49, out_features=128, bias=False)
)

In [37]:
def run_epoch(model, dataloader, optimizer=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    total_examples = 0

    for features, labels in dataloader:
        features = features.to(DEVICE)
        labels = labels.to(DEVICE)

        if training:
            optimizer.zero_grad(set_to_none=True)

        logits = model(features)
        loss = criterion(logits, labels)

        if training:
            loss.backward()
            optimizer.step()

        total_examples += labels.shape[0]
        total_loss += loss.item() * labels.shape[0]

    return total_loss / max(total_examples, 1)


best_dev_loss = float('inf')
best_checkpoint_path = CHECKPOINT_DIR / 'best.pt'
latest_checkpoint_path = CHECKPOINT_DIR / 'latest.pt'
epochs_without_improvement = 0

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = run_epoch(model, train_loader, optimizer=optimizer)
    dev_loss = run_epoch(model, dev_loader, optimizer=None)
    scheduler.step(dev_loss)

    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss': train_loss,
        'dev_loss': dev_loss,
    }
    epoch_checkpoint_path = CHECKPOINT_DIR / f'epoch_{epoch:02d}.pt'
    torch.save(checkpoint, epoch_checkpoint_path)
    torch.save(checkpoint, latest_checkpoint_path)

    improved = dev_loss < (best_dev_loss - MIN_DELTA)
    if improved:
        best_dev_loss = dev_loss
        epochs_without_improvement = 0
        torch.save(checkpoint, best_checkpoint_path)
    else:
        epochs_without_improvement += 1

    current_lr = optimizer.param_groups[0]['lr']
    print({
        'epoch': epoch,
        'train_loss': round(train_loss, 5),
        'dev_loss': round(dev_loss, 5),
        'lr': current_lr,
        'saved': str(epoch_checkpoint_path),
        'best_saved': improved,
        'epochs_without_improvement': epochs_without_improvement,
    })

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"Early stopping at epoch {epoch} (best_dev_loss={best_dev_loss:.5f})")
        break

{'epoch': 1, 'train_loss': 0.33931, 'dev_loss': 1.65678, 'lr': 0.001, 'saved': '/Users/dundale/Downloads/github/Vishing-Detection-Adversarial-Model/models/prosodic/epoch_01.pt', 'best_saved': True, 'epochs_without_improvement': 0}
{'epoch': 2, 'train_loss': 0.28862, 'dev_loss': 1.42636, 'lr': 0.001, 'saved': '/Users/dundale/Downloads/github/Vishing-Detection-Adversarial-Model/models/prosodic/epoch_02.pt', 'best_saved': True, 'epochs_without_improvement': 0}
{'epoch': 3, 'train_loss': 0.27723, 'dev_loss': 1.49281, 'lr': 0.001, 'saved': '/Users/dundale/Downloads/github/Vishing-Detection-Adversarial-Model/models/prosodic/epoch_03.pt', 'best_saved': False, 'epochs_without_improvement': 1}
{'epoch': 4, 'train_loss': 0.27236, 'dev_loss': 1.47551, 'lr': 0.001, 'saved': '/Users/dundale/Downloads/github/Vishing-Detection-Adversarial-Model/models/prosodic/epoch_04.pt', 'best_saved': False, 'epochs_without_improvement': 2}
{'epoch': 5, 'train_loss': 0.26719, 'dev_loss': 1.79291, 'lr': 0.001, 'sav

In [38]:
# Load best checkpoint and save scaler
best_checkpoint = torch.load(best_checkpoint_path, weights_only=True)
model.load_state_dict(best_checkpoint['model_state_dict'])
joblib.dump(scaler, CHECKPOINT_DIR / 'scaler.joblib')
print(f"Loaded best model from epoch {best_checkpoint['epoch']}, saved scaler")

Loaded best model from epoch 10, saved scaler


## FGSM Evaluation (Prosodic Features)

In [39]:
feature_scale = float(np.mean(np.std(X_train, axis=0)))
fgsm_epsilon = max(1e-4, 0.05 * feature_scale)
clamp_min = float(np.percentile(X_train, 1))
clamp_max = float(np.percentile(X_train, 99))

fgsm_attack = FGSMAttack(
    config=FGSMAttackConfig(
        epsilon=fgsm_epsilon,
        clamp_min=clamp_min,
        clamp_max=clamp_max,
        targeted=False,
    )
)

print({
    'epsilon': round(fgsm_epsilon, 6),
    'clamp_min': round(clamp_min, 4),
    'clamp_max': round(clamp_max, 4),
})

{'epsilon': 0.05, 'clamp_min': -2.2936, 'clamp_max': 2.6941}


In [40]:
all_labels = []
all_clean_predictions = []
all_adversarial_predictions = []
all_clean_true_probabilities = []
all_adversarial_true_probabilities = []

model.eval()
for features, labels in eval_loader:
    features = features.to(DEVICE)
    labels = labels.to(DEVICE)

    # Clean predictions
    with torch.no_grad():
        clean_logits = model(features)
        clean_probabilities = torch.softmax(clean_logits, dim=1)
        clean_predictions = clean_probabilities.argmax(dim=1)
        clean_true_probabilities = clean_probabilities.gather(1, labels.unsqueeze(1)).squeeze(1)

    # Adversarial predictions
    attack_result = fgsm_attack.generate(model=model, inputs=features, labels=labels)
    adversarial_features = attack_result.adversarial_inputs

    with torch.no_grad():
        adversarial_logits = model(adversarial_features)
        adversarial_probabilities = torch.softmax(adversarial_logits, dim=1)
        adversarial_predictions = adversarial_probabilities.argmax(dim=1)
        adversarial_true_probabilities = adversarial_probabilities.gather(1, labels.unsqueeze(1)).squeeze(1)

    all_labels.append(labels.detach().cpu())
    all_clean_predictions.append(clean_predictions.detach().cpu())
    all_adversarial_predictions.append(adversarial_predictions.detach().cpu())
    all_clean_true_probabilities.append(clean_true_probabilities.detach().cpu())
    all_adversarial_true_probabilities.append(adversarial_true_probabilities.detach().cpu())

all_labels = torch.cat(all_labels)
all_clean_predictions = torch.cat(all_clean_predictions)
all_adversarial_predictions = torch.cat(all_adversarial_predictions)
all_clean_true_probabilities = torch.cat(all_clean_true_probabilities)
all_adversarial_true_probabilities = torch.cat(all_adversarial_true_probabilities)

print({'num_samples': int(all_labels.shape[0])})

{'num_samples': 23410}


### Accuracy

In [41]:
clean_accuracy = (all_clean_predictions == all_labels).float().mean().item()
adversarial_accuracy = (all_adversarial_predictions == all_labels).float().mean().item()

print(f"Clean Accuracy: {clean_accuracy:.4f}")
print(f"Adversarial Accuracy: {adversarial_accuracy:.4f}")

Clean Accuracy: 0.5668
Adversarial Accuracy: 0.5276


### Mean Squared Confidence Degradation (MSCD)

In [42]:
delta_probabilities = all_clean_true_probabilities - all_adversarial_true_probabilities
mscd = (delta_probabilities ** 2).mean().item()

print(f"MSCD: {mscd:.6f}")

MSCD: 0.086102


### Attack Success Rate (ASR)

In [43]:
asr = (all_clean_predictions != all_adversarial_predictions).float().mean().item()

print(f"ASR: {asr:.4f}")

ASR: 0.3112
